In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from transformers import BertModel, BertTokenizer
from torch.utils.data import Dataset, DataLoader
import pandas as pd

# Load tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Dataset class
class MultiLabelDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=128):
        self.df = df
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = row['sentence']
        labels = torch.tensor([row['structure_focus'], row['usecase_focus'], row['process_focus']], dtype=torch.float)
        encoding = self.tokenizer(text, padding='max_length', truncation=True, max_length=self.max_length, return_tensors='pt')
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': labels
        }

# Model definition
class BERT_LAM(nn.Module):
    def __init__(self, num_labels=3):
        super(BERT_LAM, self).__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.label_embedding = nn.Embedding(num_labels, 768)
        self.fc = nn.Linear(768, 1)  # Fix: Output 1 value per label instead of 3
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        text_embeddings = outputs.last_hidden_state[:, 0, :]  # Shape: [batch_size, 768]
        label_embeds = self.label_embedding(torch.arange(3).to(text_embeddings.device))  # Shape: [3, 768]

        attn_weights = F.cosine_similarity(
            text_embeddings.unsqueeze(1),  # Shape: [batch_size, 1, 768]
            label_embeds.unsqueeze(0),     # Shape: [1, 3, 768]
            dim=-1
        )  # Output shape: [batch_size, 3]

        attn_applied = text_embeddings.unsqueeze(1) * attn_weights.unsqueeze(-1)  # Shape: [batch_size, 3, 768]

        logits = self.fc(attn_applied).squeeze(-1)  # Fix: Shape now [batch_size, 3]
        
        return self.sigmoid(logits)  # Final output: [batch_size, 3]


# Training function
def train_model(model, dataloader, epochs=5, lr=5e-5):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCELoss()
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(dataloader)}")

# Load data and run training
df = pd.read_csv('../type_classification-train.csv')  # Replace with actual file path
dataset = MultiLabelDataset(df, tokenizer)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

model = BERT_LAM()
train_model(model, dataloader)


C:\Users\RAYMOND\AppData\Local\pypoetry\Cache\virtualenvs\text2uml-yWP-Bm0a-py3.11\Lib\site-packages\huggingface_hub\file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
C:\Users\RAYMOND\AppData\Local\pypoetry\Cache\virtualenvs\text2uml-yWP-Bm0a-py3.11\Lib\site-packages\huggingface_hub\file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertModel: ['cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.weight', 'cls.predictions

Epoch 1/5, Loss: 0.0497924580514355
Epoch 2/5, Loss: 0.006436505769521986
Epoch 3/5, Loss: 0.0025424569300542188
Epoch 4/5, Loss: 0.001317602658599178
Epoch 5/5, Loss: 0.0007826331757871453


In [4]:
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optim.Adam(model.parameters(), lr=5e-5)
}, './model/GPT-o1.pth')

In [5]:
def predict(model, tokenizer, text):
    model.eval()  # Set to evaluation mode
    inputs = tokenizer(text, padding="max_length", truncation=True, max_length=128, return_tensors="pt")
    
    with torch.no_grad():  # Disable gradient calculation for inference
        outputs = model(inputs["input_ids"], inputs["attention_mask"])  # Shape: [1, num_labels]

    predictions = (outputs > 0.5).int()  # Convert probabilities to binary labels (threshold = 0.5)
    return predictions.squeeze().tolist()  # Convert to list

# row['structure_focus'], row['usecase_focus'], row['process_focus']

text = "For our trucks, we need to know the current odometer reading,  the gas tank capacity, and whether or not it has a working radio"
predicted_labels = predict(model, tokenizer, text)
print(predicted_labels)  # Example Output: [1, 0, 1] (Label 1 & 3 predicted as active)


[0, 0, 0]


In [6]:
from sklearn.metrics import precision_score, recall_score, f1_score

def evaluate_model(model, dataloader):
    model.eval()  # Set model to evaluation mode
    
    all_targets = []
    all_predictions = []
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].cpu().numpy()  # Ground truth labels
            
            outputs = model(input_ids, attention_mask)  # Predicted probabilities
            predictions = (outputs > 0.5).int().cpu().numpy()  # Convert to binary labels
            
            all_targets.extend(labels)
            all_predictions.extend(predictions)

    # Compute Precision, Recall, and F1-score
    precision = precision_score(all_targets, all_predictions, average="micro")  # or "macro"
    recall = recall_score(all_targets, all_predictions, average="micro")
    f1 = f1_score(all_targets, all_predictions, average="micro")

    print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1-score: {f1:.4f}")
    return precision, recall, f1

evaluate_model(model, dataloader)  # This will print and return precision, recall, and F1-score


Precision: 0.0000, Recall: 0.0000, F1-score: 0.0000


C:\Users\RAYMOND\AppData\Local\pypoetry\Cache\virtualenvs\text2uml-yWP-Bm0a-py3.11\Lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\RAYMOND\AppData\Local\pypoetry\Cache\virtualenvs\text2uml-yWP-Bm0a-py3.11\Lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\RAYMOND\AppData\Local\pypoetry\Cache\virtualenvs\text2uml-yWP-Bm0a-py3.11\Lib\site-packages\sklearn\metrics\_classification.py:1609: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.


(0.0, 0.0, 0.0)